In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd

dataset = pd.read_csv('reddit_preprocessing.csv')
cleaned_dataset = dataset.dropna()

X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned  # ✅ FIX: added stratify
)

tfidf_cleaned = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)
X_train_cleaned_tfidf = tfidf_cleaned.fit_transform(X_train_cleaned)
X_test_cleaned_tfidf = tfidf_cleaned.transform(X_test_cleaned)

lightgbm_model = LGBMClassifier(
    objective='multiclass',       # ✅ FIX: 'objectnum_class' → 'objective' with correct value 'multiclass'
    num_class=3,                  # ✅ FIX: 'objectnum_class=3' was a typo merge — split into separate param
    metric='multi_logloss',
    is_unbalance=True,
    class_weight='balanced',      # ✅ FIX: 'balance' → 'balanced'
    reg_alpha=0.1,
    reg_lambda=0.1,
    learning_rate=0.08081298097796712,
    n_estimators=367,
    max_depth=20,
    verbose=-1                    # ✅ FIX: suppress noisy LightGBM training logs
)

logreg_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    multi_class='multinomial'
)

knn_meta_learner = KNeighborsClassifier(n_neighbors=5)

estimators = [('lightgbm', lightgbm_model), ('logreg', logreg_model)]

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=knn_meta_learner,
    cv=5
)

stacking_model.fit(X_train_cleaned_tfidf, y_train_cleaned)
y_pred_cleaned = stacking_model.predict(X_test_cleaned_tfidf)
print(classification_report(y_test_cleaned, y_pred_cleaned))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier wa

              precision    recall  f1-score   support

          -1       0.81      0.75      0.78      1650
           0       0.86      0.95      0.90      2522
           1       0.89      0.86      0.88      3154

    accuracy                           0.86      7326
   macro avg       0.85      0.85      0.85      7326
weighted avg       0.86      0.86      0.86      7326

